# 4.0 · Transformación

> Nominal → one-hot. Ordinal solo con jerarquía real. **Nunca escalar binarias**. Fechas/duración no entran a X.

In [ ]:
from pathlib import Path
import sys

def _carpeta_refactor() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'lib' / 'config.py').exists(): return p
        if (p / 'notebooks' / 'refactor' / 'lib' / 'config.py').exists(): return p / 'notebooks' / 'refactor'
    raise RuntimeError('No encontrado')
RAIZ=_carpeta_refactor(); sys.path.insert(0,str(RAIZ))
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from lib import carga, categorias, competencias, config, periodo
print('ok')


In [ ]:
df = carga.leer(config.CON_OUTLIERS)
df['duracion']=pd.to_numeric(periodo.duracion_observada(df), errors='coerce')
df['nivel_ordinal']=pd.to_numeric(categorias.nivel_ordinal(df), errors='coerce')
df['nivel_no_reportado']=categorias.nivel_no_reportado(df)
df['isco_mayor']=categorias.isco_mayor(df)
df['n_skills_essential']=pd.to_numeric(df['n_skills_essential'], errors='coerce')
df['veces_ese_oficio']=pd.to_numeric(df['veces_ese_oficio'], errors='coerce')
df['es_rescatado']=(df['es_rescatado'].astype(str).str.lower()=='true')
df['es_vigente']=(df['es_vigente'].astype(str).str.lower()=='true')
print(df.shape)


In [ ]:
binarias = ['nivel_no_reportado','es_rescatado','es_vigente']
X = pd.get_dummies(df[['nivel_ordinal','n_skills_essential','veces_ese_oficio','isco_mayor','emparejado']+binarias].copy(), columns=['isco_mayor','emparejado'], dummy_na=False)
X[binarias]=X[binarias].astype(int)
scaler=StandardScaler()
for c in ['n_skills_essential','veces_ese_oficio']:
    X[c]=X[c].fillna(X[c].median())
X[['n_skills_essential','veces_ese_oficio']]=scaler.fit_transform(X[['n_skills_essential','veces_ese_oficio']])
print('X shape:', X.shape)
